# Qwen-27B Prompt-Conditioned Route Relocation Audit

The structured-calculation experiment produced no physics-specific route repair at the two frozen options/answer sites.

[实验设置与运行说明](../../../docs/experiment_notes/09_appendix_factor_routes/fccd3dc765a6779b.md)


In [ ]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 17 tracked cells.')


## 1. Setup


In [ ]:
%%cell_progress 1/17 Mount Drive and locate VisualTorque
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (
        path for path in PROJECT_CANDIDATES
        if (path / 'toolkit' / 'pyproject.toml').is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


In [ ]:
%%cell_progress 2/17 Install and verify the multimodal environment
import importlib
import os

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import subprocess
import sys

cell_stage('Removing unused TorchAudio')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
    check=False,
)
cell_stage('Installing pinned inference and analysis dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.14.0', 'accelerate>=1.10', 'sentencepiece',
    'protobuf', 'huggingface_hub>=0.34', 'hf_xet>=1.1', 'tqdm>=4.66',
    'pandas>=2.2', 'matplotlib>=3.8', 'scikit-learn>=1.5',
])

cell_stage('Registering the live VisualTorque source tree')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
assert (PACKAGE_ROOT / '__init__.py').is_file(), PACKAGE_ROOT
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if (
        module_name == 'visual_torque_toolkit'
        or module_name.startswith('visual_torque_toolkit.')
    ):
        del sys.modules[module_name]
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
import visual_torque_toolkit
from IPython.display import display
from transformers import AutoModelForMultimodalLM, AutoProcessor
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter

required_features = (
    '_template', '_to_device', '_candidate_plan', '_append_text_prefix',
)
assert all(
    hasattr(HuggingFaceMultimodalAdapter, name)
    for name in required_features
)
toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents, toolkit_path
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
print('Toolkit source:', toolkit_path)


In [ ]:
%%cell_progress 3/17 Verify A100 and Hugging Face access
cell_stage('Checking CUDA device')
assert torch.cuda.is_available(), 'Select an A100 GPU runtime.'
print(torch.cuda.get_device_name(0))
print('CUDA memory (GB):', round(
    torch.cuda.get_device_properties(0).total_memory / 2**30, 1
))

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    token = userdata.get('HF_TOKEN')
    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login complete.')
    else:
        print('HF_TOKEN is not set. Public checkpoints may still work.')
except Exception as exc:
    print('Hugging Face login skipped:', exc)


In [ ]:
%%cell_progress 4/17 Lock the route-relocation protocol
cell_stage('Freezing model, support, exact anchor, scan grid, and global test')
import hashlib
import json
from datetime import datetime, timezone

MODEL_ID = 'Qwen/Qwen3.5-27B'
MODEL_REVISION = '373612cb4f9288fcf550b526d86d85febedd9801'
MODEL_SLUG = 'Qwen__Qwen3.5-27B__373612cb'
EXPECTED_LAYER_COUNT = 64
MODEL_DTYPE = 'bfloat16'
DEVICE_MAP = 'auto'

SEED = 20261103
PHASE_OPTION_SEED = 20261021
EXPECTED_FAMILIES = 20
FAMILIES_PER_CONFLICT_TYPE = 10
MAX_CUE_DONOR_MARGIN_GAP = 0.15

TASKS = ('net_torque', 'outcome')
PROMPT_STYLES = ('phase_original', 'generic_deliberation', 'structured_physics')
SCENE_ROLES = ('base', 'mirror_source', 'cue_source')
INTERVENTIONS = ('self_base', 'mirror_source', 'cue_source')
SCAN_LAYERS = (8, 16, 24, 32, 40, 48, 56, 63)
SCAN_SITES = tuple(
    (layer, 'answer_instruction_tokens', f'l{layer}_instruction')
    for layer in SCAN_LAYERS
) + tuple(
    (layer, 'options_tokens', f'l{layer}_options')
    for layer in SCAN_LAYERS
) + ((63, 'final_prompt_token', 'l63_answer_anchor'),)
DIRECT_SITES = SCAN_SITES
PRIMARY_SITES = tuple(site[2] for site in SCAN_SITES)
PRIOR_FROZEN_SITES = ('l48_options', 'l63_answer_anchor')
CAPTURE_LAYERS = SCAN_LAYERS
REQUIRED_SCOPES = (
    'valid_tokens', 'answer_instruction_tokens',
    'options_tokens', 'final_prompt_token',
)

PATCH_CONDITION_BATCH_SIZE = 1
MIN_STRUCTURED_ACCURACY = 0.50
PARITY_TOLERANCE = 1e-3
BOOTSTRAP_REPLICATES = 20000
PERMUTATION_REPLICATES = 20000
ALPHA = 0.05
IMAGE_TOKEN_ID_OVERRIDE = None
SMOKE_MAX_SELF_REPLAY_ERROR = 0.15

PHASE_ROOT = (
    PROJECT_ROOT / 'results' / 'qwen27b_conditional_strategy_phase_map_v1'
)
PHASE_MANIFEST_PATH = PHASE_ROOT / 'strategy_family_manifest.jsonl'
PHASE_SHARD_DIR = PHASE_ROOT / 'models' / MODEL_SLUG / 'family_shards'
PHASE_DECISION_PATH = (
    PHASE_ROOT / 'models' / MODEL_SLUG / 'analysis'
    / 'conditional_strategy_decision.json'
)
ELICITATION_DECISION_PATH = (
    PROJECT_ROOT / 'results' / 'qwen27b_calculation_elicitation_v2'
    / 'models' / MODEL_SLUG / 'analysis'
    / 'calculation_elicitation_decision.json'
)
for required in (
    PHASE_MANIFEST_PATH, PHASE_SHARD_DIR,
    PHASE_DECISION_PATH, ELICITATION_DECISION_PATH,
):
    assert required.exists(), f'Missing prerequisite: {required}'
phase_decision = json.loads(PHASE_DECISION_PATH.read_text(encoding='utf-8'))
elicitation_decision = json.loads(
    ELICITATION_DECISION_PATH.read_text(encoding='utf-8')
)
assert phase_decision['decision'] == 'shortcut_sensitive_without_adaptive_route_switch'
assert elicitation_decision['elicitation_signal'] is False

EXPERIMENT_ROOT = (
    PROJECT_ROOT / 'results' / 'qwen27b_route_relocation_audit_v1'
)
MANIFEST_PATH = EXPERIMENT_ROOT / 'route_relocation_manifest.jsonl'
MANIFEST_LOCK_PATH = EXPERIMENT_ROOT / 'manifest_lock.json'
PROTOCOL_LOCK_PATH = EXPERIMENT_ROOT / 'claim_registry.json'
OUTPUT_ROOT = EXPERIMENT_ROOT / 'models' / MODEL_SLUG
SHARD_DIR = OUTPUT_ROOT / 'block_shards'
ANALYSIS_DIR = OUTPUT_ROOT / 'analysis'
FIGURE_DIR = ANALYSIS_DIR / 'figures'
for path in (
    EXPERIMENT_ROOT, OUTPUT_ROOT, SHARD_DIR, ANALYSIS_DIR, FIGURE_DIR,
):
    path.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return 'sha256:' + digest.hexdigest()

def write_json_atomic(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False) + '\n',
        encoding='utf-8',
    )
    temporary.replace(path)

protocol_identity = {
    'schema_version': 'qwen27b-route-relocation-audit-1.0',
    'analysis_status': 'confirmatory_global_depth_token_scan',
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'phase_decision_sha256': sha256_file(PHASE_DECISION_PATH),
    'elicitation_decision_sha256': sha256_file(ELICITATION_DECISION_PATH),
    'phase_manifest_sha256': sha256_file(PHASE_MANIFEST_PATH),
    'support': (
        '20 frozen phase-map high-margin conflict families; '
        '10 weight-conflict and 10 distance-conflict; outcomes balanced'
    ),
    'prompt_styles': list(PROMPT_STYLES),
    'phase_option_seed': PHASE_OPTION_SEED,
    'tasks': list(TASKS),
    'roles': list(SCENE_ROLES),
    'interventions': list(INTERVENTIONS),
    'scan_layers': list(SCAN_LAYERS),
    'scan_sites': [list(site) for site in SCAN_SITES],
    'primary_estimand': (
        'At every frozen task/site, paired structured_physics minus '
        'generic_deliberation route-index gain, with a positive '
        'structured_physics minus phase_original gain required at the same site.'
    ),
    'global_multiplicity': (
        'one-sided shared-family max-statistic sign-flip correction across '
        'all 34 frozen task-by-site structured-minus-generic tests'
    ),
    'parity_gate': (
        'phase_original centered candidate logprobs reproduce the saved '
        f'phase-map forward pass within {PARITY_TOLERANCE}'
    ),
    'claim_gate': (
        'structured accuracy >=0.50 for both tasks; parity passes; '
        'structured-minus-generic bootstrap CI>0 and maxT p<0.05; '
        'structured-minus-original CI>0; both LOO minima>0; '
        'weight- and distance-conflict gains positive; '
        'left- and right-outcome gains positive'
    ),
    'nonconditioning_rule': (
        'correctness is descriptive only; no family, layer, scope, or task '
        'is selected from model outputs'
    ),
    'bootstrap_replicates': BOOTSTRAP_REPLICATES,
    'permutation_replicates': PERMUTATION_REPLICATES,
    'seed': SEED,
    'interpretation_limit': (
        'A positive site supports prompt-conditioned relocation of a '
        'sufficient residual route. A null does not exclude a route spread '
        'across several layers or components.'
    ),
}
if PROTOCOL_LOCK_PATH.is_file():
    previous = json.loads(PROTOCOL_LOCK_PATH.read_text(encoding='utf-8'))
    assert previous.get('identity') == protocol_identity, (
        'Protocol changed. Use a new versioned experiment directory.'
    )
else:
    write_json_atomic(PROTOCOL_LOCK_PATH, {
        'identity': protocol_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })

print('Model:', MODEL_ID, '@', MODEL_REVISION)
print('Frozen families:', EXPECTED_FAMILIES)
print('Prompt styles:', PROMPT_STYLES)
print('Scan layers:', SCAN_LAYERS)
print('Sites / primary tests:', len(SCAN_SITES), len(SCAN_SITES) * len(TASKS))
print('Output:', OUTPUT_ROOT)


In [ ]:
%%cell_progress 5/17 Prepare Dataset v1 on local SSD
cell_stage('Checking local and Drive dataset caches')
from zipfile import ZipFile

REQUIRED_DATASET_FILES = {
    'dataset_config.json', 'scenes.jsonl', 'counterfactual_pairs.jsonl',
    'questions.jsonl', 'splits.json', 'statistics.json',
}
DATASET_DIR = Path('/content/visualtorque_dataset_v1')
DATASET_ARCHIVE = PROJECT_ROOT / 'data' / 'dataset_v1.zip'
LEGACY_DATASET_DIR = PROJECT_ROOT / 'data' / 'dataset_v1'

def dataset_complete(path):
    return path.is_dir() and all(
        (path / name).is_file() for name in REQUIRED_DATASET_FILES
    )

if not dataset_complete(DATASET_DIR):
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    if DATASET_ARCHIVE.is_file():
        cell_stage('Extracting Dataset v1 archive')
        with ZipFile(DATASET_ARCHIVE, 'r') as bundle:
            members = [m for m in bundle.infolist() if not m.is_dir()]
            root = DATASET_DIR.resolve()
            for member in tqdm(members, desc='Extracting Dataset v1', unit='file'):
                target = (DATASET_DIR / member.filename).resolve()
                if root not in target.parents:
                    raise ValueError(f'Unsafe archive path: {member.filename}')
                bundle.extract(member, DATASET_DIR)
    elif dataset_complete(LEGACY_DATASET_DIR):
        DATASET_DIR = LEGACY_DATASET_DIR
    else:
        raise FileNotFoundError(
            'Run the Dataset v1 build/cache cell in the behavior notebook first.'
        )
assert dataset_complete(DATASET_DIR)
print('Dataset:', DATASET_DIR)


## 2. Frozen support and exact phase anchor


In [ ]:
%%cell_progress 6/17 Restore the frozen conflict support and phase baselines
cell_stage('Loading the phase manifest, phase shards, and Dataset v1 examples')
from collections import Counter, defaultdict

from visual_torque_toolkit.behavior import (
    BehaviorRunConfig, PromptCondition, load_behavior_examples,
)

def read_jsonl(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def write_jsonl_atomic(path, rows):
    temporary = Path(path).with_suffix(Path(path).suffix + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(row, sort_keys=True) + '\n')
    temporary.replace(path)

def analysis_key(value):
    return hashlib.sha256(f'{SEED}:{value}'.encode('utf-8')).hexdigest()

def phase_option_key(value):
    return hashlib.sha256(
        f'{PHASE_OPTION_SEED}:{value}'.encode('utf-8')
    ).hexdigest()

scene_records = read_jsonl(DATASET_DIR / 'scenes.jsonl')
phase_rows = read_jsonl(PHASE_MANIFEST_PATH)
assert len(phase_rows) == 80
selected = [
    row for row in phase_rows
    if row['cue_class'] == 'conflict' and row['margin_bin'] == 'high'
]
selected.sort(key=lambda row: row['family_id'])
assert len(selected) == EXPECTED_FAMILIES
assert Counter(row['cue_type'] for row in selected) == {
    'weight_conflict': FAMILIES_PER_CONFLICT_TYPE,
    'distance_conflict': FAMILIES_PER_CONFLICT_TYPE,
}
assert Counter(row['base_outcome'] for row in selected) == {
    'left': EXPECTED_FAMILIES // 2,
    'right': EXPECTED_FAMILIES // 2,
}
selected_scene_ids = {
    str(row[key]) for row in selected
    for key in ('base_scene_id', 'mirror_scene_id', 'cue_scene_id')
}
assert len(selected_scene_ids) == 3 * EXPECTED_FAMILIES

manifest_identity = {
    'schema_version': 'route-relocation-manifest-1.0',
    'dataset_scenes_sha256': sha256_file(DATASET_DIR / 'scenes.jsonl'),
    'source_phase_manifest_sha256': sha256_file(PHASE_MANIFEST_PATH),
    'selection_rule': 'cue_class=conflict and margin_bin=high',
    'families': EXPECTED_FAMILIES,
    'family_ids': [row['family_id'] for row in selected],
    'model_outputs_consulted': False,
}
if MANIFEST_LOCK_PATH.is_file():
    previous = json.loads(MANIFEST_LOCK_PATH.read_text(encoding='utf-8'))
    assert previous.get('identity') == manifest_identity
    assert read_jsonl(MANIFEST_PATH) == selected
else:
    write_jsonl_atomic(MANIFEST_PATH, selected)
    write_json_atomic(MANIFEST_LOCK_PATH, {
        'identity': manifest_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })
relocation_manifest = read_jsonl(MANIFEST_PATH)

phase_baselines = {}
for path in sorted(PHASE_SHARD_DIR.glob('shard_family_*.json')):
    payload = json.loads(path.read_text(encoding='utf-8'))
    phase_baselines[str(payload['family_id'])] = payload['baselines']
assert all(row['family_id'] in phase_baselines for row in relocation_manifest)

manifest_frame = pd.DataFrame([{
    key: value for key, value in row.items()
    if key not in {'task_values', 'base_shape', 'mirror_shape', 'cue_shape'}
} for row in relocation_manifest])
manifest_frame.to_csv(EXPERIMENT_ROOT / 'route_relocation_manifest.csv', index=False)
display(manifest_frame.groupby([
    'cue_type', 'base_outcome'
]).size().rename('n').reset_index())
print('Frozen families / scenes:', len(relocation_manifest), len(selected_scene_ids))
print('Recovered phase baselines:', len(phase_baselines))
print('Cue donors margin matched:', int(
    (manifest_frame['cue_source_margin_gap'] <= MAX_CUE_DONOR_MARGIN_GAP).sum()
), '/', len(manifest_frame))

example_config = BehaviorRunConfig(
    dataset_dir=DATASET_DIR,
    output_dir=EXPERIMENT_ROOT / '_unused_behavior_loader',
    model_id=MODEL_ID,
    model_revision=MODEL_REVISION,
    conditions=(PromptCondition.IMAGE_RULE,),
    question_kinds=('outcome',),
    splits=('probe_train', 'probe_dev', 'probe_test'),
    batch_size=1, generation_every=0, seed=SEED,
)
outcome_examples = tuple(load_behavior_examples(example_config))
outcome_by_scene = {
    str(example.scene_id): example for example in outcome_examples
    if example.variant == 'permutation_0'
}
assert len(outcome_by_scene) == len(scene_records)
print('Pinned Dataset v1 image examples:', len(outcome_by_scene))


In [ ]:
%%cell_progress 7/17 Load Qwen3.5-27B and resolve frozen decoder layers
cell_stage('Loading pinned 27B weights')
import gc
if 'adapter' in globals():
    del adapter
    gc.collect()
    torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

adapter = HuggingFaceMultimodalAdapter(
    MODEL_ID, revision=MODEL_REVISION,
    dtype=MODEL_DTYPE, device_map=DEVICE_MAP,
)
display(adapter.metadata)

def resolve_module_path(root, path):
    value = root
    for component in path.split('.'):
        value = getattr(value, component)
    return value

def discover_decoder_layers(model):
    candidates = (
        'model.language_model.layers', 'model.text_model.layers',
        'language_model.model.layers', 'model.model.layers', 'model.layers',
    )
    for path in candidates:
        try:
            layers = resolve_module_path(model, path)
        except AttributeError:
            continue
        if isinstance(layers, torch.nn.ModuleList) and len(layers) >= 8:
            return path, layers
    raise RuntimeError('Could not locate decoder layers.')

LAYER_PATH, DECODER_LAYERS = discover_decoder_layers(adapter.model)
assert len(DECODER_LAYERS) == EXPECTED_LAYER_COUNT, (
    LAYER_PATH, len(DECODER_LAYERS), EXPECTED_LAYER_COUNT
)
assert all(0 <= layer < len(DECODER_LAYERS) for layer in CAPTURE_LAYERS)
device_map = getattr(adapter.model, 'hf_device_map', {})
disk_modules = [name for name, device in device_map.items() if device == 'disk']
assert not disk_modules, ('Disk offload is too slow:', disk_modules[:5])
print('Layer path:', LAYER_PATH)
print('Frozen layers:', CAPTURE_LAYERS)
print('Allocated GPU GB:', round(torch.cuda.memory_allocated() / 2**30, 2))


## 3. Globally corrected depth × token-scope scan


In [ ]:
%%cell_progress 8/17 Define exact phase, generic, and structured prompts and masks
cell_stage('Restoring phase option order and defining within-style token scopes')
import math
from dataclasses import replace

RULE_TEXT = (
    'Each square is one unit of weight. Distance is measured in tick '
    "intervals from the central pivot. A stack's turning effect is its "
    'number of squares multiplied by its distance. Compare the total '
    'turning effects on the two sides.'
)
STYLE_TEXT = {
    'generic_deliberation': (
        'Before answering, inspect the full image in a fixed sequence, '
        'consider every visible detail, compare all alternatives, and '
        'double-check the choice. Perform these checks silently.'
    ),
    'structured_physics': (
        'Before answering, silently identify the units and pivot distance on '
        'each side, multiply units by distance to obtain both side torques, '
        'subtract right torque from left torque, and use that result.'
    ),
}
OUTCOME_TEXT = {
    'left': 'left side', 'balanced': 'balanced', 'right': 'right side',
}

def display_value(task, value):
    return OUTCOME_TEXT[str(value)] if task == 'outcome' else str(int(value))

def task_question(task):
    return {
        'net_torque': 'What is left torque minus right torque?',
        'outcome': 'Which side will rotate downward?',
    }[task]

def rotated(values, row, task):
    values = list(values)
    assert len(values) == 3 and len(set(values)) == 3, (task, values)
    offset = int(phase_option_key(
        f"options:{row['family_id']}:{task}"
    )[:8], 16) % 3
    return tuple(values[offset:] + values[:offset])

def task_semantics(row, task):
    if task == 'outcome':
        return rotated(['left', 'balanced', 'right'], row, task)
    base_value = int(row['task_values']['net_torque']['base'])
    mirror_value = int(row['task_values']['net_torque']['mirror_source'])
    assert base_value == -mirror_value and base_value != 0
    return rotated([base_value, mirror_value, 0], row, task)

def custom_prompt(task, semantics, prompt_style):
    labels = ('A', 'B', 'C')
    options = '\n'.join(
        f'{label}. {display_value(task, value)}'
        for label, value in zip(labels, semantics)
    )
    if prompt_style == 'phase_original':
        answer_instruction = (
            'Do not explain your reasoning. Reply with exactly one option '
            'letter and nothing else. Valid replies are: A, B, C.'
        )
    else:
        answer_instruction = (
            STYLE_TEXT[prompt_style]
            + ' Reply with exactly one option letter and nothing else. '
            'Valid replies are: A, B, C.'
        )
    return '\n\n'.join([
        'Solve this balance-scale question using only the supplied scene.',
        RULE_TEXT,
        'Question: ' + task_question(task),
        'Options:\n' + options,
        answer_instruction,
    ])

def make_task_bundle(row, task, prompt_style):
    semantics = task_semantics(row, task)
    values = {role: row['task_values'][task][role] for role in SCENE_ROLES}
    if task == 'net_torque':
        values = {key: int(value) for key, value in values.items()}
    else:
        values = {key: str(value) for key, value in values.items()}
    scene_ids = {
        'base': str(row['base_scene_id']),
        'mirror_source': str(row['mirror_scene_id']),
        'cue_source': str(row['cue_scene_id']),
    }
    labels = ('A', 'B', 'C')
    prompt = custom_prompt(task, semantics, prompt_style)
    examples = {}
    for role in SCENE_ROLES:
        value = values[role]
        assert value in semantics
        original = outcome_by_scene[scene_ids[role]]
        gold_index = semantics.index(value)
        examples[role] = replace(
            original,
            example_id=(
                f"relocation:{row['family_id']}:{prompt_style}:{task}:{role}"
            ),
            question_id=f"relocation_{row['family_id']}_{prompt_style}_{task}",
            question_kind=task,
            variant=f'route_relocation_{prompt_style}',
            split='frozen_route_relocation_support',
            problem_type='prompt_conditioned_route_relocation',
            prompt=prompt,
            labels=labels,
            option_text=tuple(display_value(task, value) for value in semantics),
            option_semantics=semantics,
            gold_label=labels[gold_index],
            gold_semantic=value,
        )
    assert values['base'] == values['cue_source']
    assert values['base'] != values['mirror_source']
    return {
        'task': task, 'prompt_style': prompt_style,
        'semantics': semantics, 'values': values, 'examples': examples,
    }

def output_hidden_tensor(output):
    if torch.is_tensor(output):
        hidden = output
    elif isinstance(output, (tuple, list)) and output and torch.is_tensor(output[0]):
        hidden = output[0]
    elif hasattr(output, 'last_hidden_state') and torch.is_tensor(
        output.last_hidden_state
    ):
        hidden = output.last_hidden_state
    else:
        raise TypeError(f'Unsupported output: {type(output).__name__}')
    assert hidden.ndim == 3
    return hidden

def replace_output_hidden(output, modified):
    if torch.is_tensor(output):
        return modified
    if isinstance(output, tuple):
        return (modified,) + output[1:]
    if isinstance(output, list):
        return [modified] + output[1:]
    raise TypeError(type(output).__name__)

def prepare_examples(examples):
    examples = tuple(examples)
    plan = adapter._candidate_plan(examples[0])
    if plan is None:
        raise RuntimeError('A/B/C labels must be single-token candidates.')
    for example in examples[1:]:
        assert adapter._candidate_plan(example) == plan
    fixed_prefix, token_by_label = plan
    inputs = adapter._append_text_prefix(
        adapter._to_device(adapter._template(
            [example.messages() for example in examples],
            add_generation_prompt=True,
        )),
        fixed_prefix,
    )
    return {
        'examples': examples, 'inputs': inputs,
        'token_by_label': token_by_label,
    }

def discover_masks(prepared):
    assert len(prepared['examples']) == 1
    example = prepared['examples'][0]
    ids = prepared['inputs']['input_ids'][0].detach().cpu()
    attention = prepared['inputs']['attention_mask'][0].detach().cpu().bool()
    tokenizer = getattr(adapter.processor, 'tokenizer', adapter.processor)
    tokens = tokenizer.convert_ids_to_tokens(ids.tolist())
    image_ids = set()
    if IMAGE_TOKEN_ID_OVERRIDE is not None:
        values = (
            IMAGE_TOKEN_ID_OVERRIDE
            if isinstance(IMAGE_TOKEN_ID_OVERRIDE, (list, tuple))
            else [IMAGE_TOKEN_ID_OVERRIDE]
        )
        image_ids.update(int(value) for value in values)
    for token_id, token_text in zip(ids.tolist(), tokens):
        lower = str(token_text).lower()
        if any(marker in lower for marker in ('image', 'vision', '<img', 'img>')):
            image_ids.add(int(token_id))
    image = torch.tensor(
        [int(token_id) in image_ids for token_id in ids.tolist()],
        dtype=torch.bool,
    ) & attention
    assert bool(image.any()), 'No image tokens detected.'
    final = torch.zeros_like(attention)
    final[int(torch.where(attention)[0][-1])] = True
    all_text = attention & ~image & ~final

    encoding = tokenizer(
        example.prompt, add_special_tokens=False, return_offsets_mapping=True,
    )
    prompt_ids = list(encoding['input_ids'])
    offsets = list(encoding['offset_mapping'])
    id_list = ids.tolist()
    starts = [
        start for start in range(len(ids) - len(prompt_ids) + 1)
        if id_list[start:start + len(prompt_ids)] == prompt_ids
    ]
    assert len(starts) == 1, len(starts)
    prompt_start = starts[0]
    sections = example.prompt.split('\n\n')
    assert len(sections) == 5
    names = (
        'intro_tokens', 'rule_tokens', 'question_tokens',
        'options_tokens', 'answer_instruction_tokens',
    )
    masks = {
        'valid_tokens': attention, 'image_tokens': image,
        'all_text_except_final': all_text, 'final_prompt_token': final,
    }
    spans = []
    cursor = 0
    for name, section in zip(names, sections):
        start = example.prompt.find(section, cursor)
        end = start + len(section)
        cursor = end
        spans.append((name, start, end))
        masks[name] = torch.zeros_like(attention)
    for offset_index, (token_start, token_end) in enumerate(offsets):
        overlaps = [
            (max(0, min(token_end, end) - max(token_start, start)), name)
            for name, start, end in spans
        ]
        overlap, name = max(overlaps)
        full_index = prompt_start + offset_index
        if overlap > 0 and bool(all_text[full_index]):
            masks[name][full_index] = True
    assert all(bool(masks[scope].any()) for scope in REQUIRED_SCOPES)
    return masks

def assert_aligned(prepared_by_name):
    names = list(prepared_by_name)
    reference = prepared_by_name[names[0]]
    reference_masks = discover_masks(reference)
    for name in names[1:]:
        current = prepared_by_name[name]
        for key in ('input_ids', 'attention_mask'):
            assert torch.equal(reference['inputs'][key], current['inputs'][key]), (
                name, key
            )
        masks = discover_masks(current)
        for scope in REQUIRED_SCOPES:
            assert torch.equal(reference_masks[scope], masks[scope]), (
                name, scope
            )
    return reference_masks

print('Exact phase anchor and prompt-specific token masks ready.')


In [ ]:
%%cell_progress 9/17 Define capture and single-example route-relocation interventions
cell_stage('Defining candidate scoring and grouped decoder-output interventions')

def score_records(prepared, logits):
    logprobs = torch.log_softmax(logits[:, -1, :].float(), dim=-1)
    records = []
    for row_index, example in enumerate(prepared['examples']):
        scores = [
            float(logprobs[
                row_index, prepared['token_by_label'][label]
            ].detach().cpu())
            for label in example.labels
        ]
        maximum = max(scores)
        weights = [math.exp(value - maximum) for value in scores]
        probabilities = [value / sum(weights) for value in weights]
        predicted_index = int(np.argmax(scores))
        records.append({
            'scene_id': str(example.scene_id),
            'example_id': str(example.example_id),
            'task': str(example.question_kind),
            'candidate_semantics': list(example.option_semantics),
            'candidate_logprobs': scores,
            'candidate_probabilities': probabilities,
            'predicted_semantic': example.option_semantics[predicted_index],
            'gold_semantic': example.gold_semantic,
            'correct': example.option_semantics[predicted_index] == example.gold_semantic,
        })
    return records

def forward_capture(prepared):
    hidden = {}
    handles = []
    for layer_index in CAPTURE_LAYERS:
        def hook(_module, _inputs, output, layer_index=layer_index):
            hidden[layer_index] = output_hidden_tensor(output)[0].detach().cpu().clone()
        handles.append(DECODER_LAYERS[layer_index].register_forward_hook(hook))
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        for handle in handles:
            handle.remove()
    return score_records(prepared, output.logits)[0], hidden

def forward_intervention_chunk(prepared, conditions):
    assert len(prepared['examples']) == len(conditions)
    layer_indices = {int(condition['layer_index']) for condition in conditions}
    assert len(layer_indices) == 1, layer_indices
    layer_index = next(iter(layer_indices))
    audit = {}

    def hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        donor = torch.stack([
            condition['intervention_hidden'] for condition in conditions
        ]).to(hidden.device, hidden.dtype)
        masks = torch.stack([
            condition['patch_mask'] for condition in conditions
        ]).to(hidden.device)
        assert donor.shape == hidden.shape, (donor.shape, hidden.shape)
        assert masks.shape == hidden.shape[:2], (masks.shape, hidden.shape)
        modified = hidden.clone()
        modified[masks] = donor[masks]
        audit['patched_tokens'] = [int(mask.sum().item()) for mask in masks]
        return replace_output_hidden(output, modified)

    handle = DECODER_LAYERS[layer_index].register_forward_hook(hook)
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        handle.remove()
    records = score_records(prepared, output.logits)
    assert len(records) == len(conditions)
    for record, condition, count in zip(
        records, conditions, audit['patched_tokens']
    ):
        record.update({
            key: value for key, value in condition.items()
            if key not in {'intervention_hidden', 'patch_mask'}
        })
        record['patched_tokens'] = count
    return records

def centered_logprob_error(left, right):
    left = np.asarray(left, dtype=float)
    right = np.asarray(right, dtype=float)
    left -= left.mean()
    right -= right.mean()
    return float(np.max(np.abs(left - right)))

print('Capture and route-relocation intervention scorers ready.')


In [ ]:
%%cell_progress 10/17 Run phase parity, alignment, and replay smoke tests
cell_stage('Checking exact phase prompt, masks, layer capture, and self replay')

smoke_row = relocation_manifest[0]
smoke_errors = []
prompt_lengths = {}
phase_parity_smoke = []
for prompt_style in PROMPT_STYLES:
    for task in TASKS:
        bundle = make_task_bundle(smoke_row, task, prompt_style)
        prepared = {
            role: prepare_examples([example])
            for role, example in bundle['examples'].items()
        }
        masks = assert_aligned(prepared)
        prompt_lengths[(prompt_style, task)] = int(
            prepared['base']['inputs']['attention_mask'].sum()
        )
        base_score, base_hidden = forward_capture(prepared['base'])
        assert set(base_hidden) == set(SCAN_LAYERS)
        if prompt_style == 'phase_original':
            prior = phase_baselines[smoke_row['family_id']][task]['scores']['base']
            assert base_score['candidate_semantics'] == prior['candidate_semantics']
            parity = centered_logprob_error(
                base_score['candidate_logprobs'], prior['candidate_logprobs']
            )
            phase_parity_smoke.append((task, parity))
        self_condition = {
            'family_id': smoke_row['family_id'],
            'prompt_style': prompt_style, 'task': task,
            'condition': 'self_base_smoke', 'intervention': 'self_base',
            'layer_index': 48, 'scope': 'answer_instruction_tokens',
            'site_label': 'l48_instruction',
            'intervention_hidden': base_hidden[48],
            'patch_mask': masks['answer_instruction_tokens'],
        }
        self_record = forward_intervention_chunk(
            prepared['base'], [self_condition]
        )[0]
        self_error = centered_logprob_error(
            self_record['candidate_logprobs'], base_score['candidate_logprobs']
        )
        assert np.isfinite(self_error)
        assert self_record['patched_tokens'] == int(
            masks['answer_instruction_tokens'].sum()
        )
        smoke_errors.append((prompt_style, task, self_error))

for task in TASKS:
    generic_length = prompt_lengths[('generic_deliberation', task)]
    structured_length = prompt_lengths[('structured_physics', task)]
    assert abs(generic_length - structured_length) <= 24
assert max(value for _, value in phase_parity_smoke) <= PARITY_TOLERANCE, (
    phase_parity_smoke
)
print('Phase-original parity smoke:', phase_parity_smoke)
print('Centered self-replay audit:', smoke_errors)
print('Prompt token lengths:', prompt_lengths)


In [ ]:
%%cell_progress 11/17 Capture or resume the frozen depth × token grid
cell_stage('Running block-sharded route scan; follow family and site bars')

RECORDS_PER_BLOCK = len(SCAN_SITES) * len(INTERVENTIONS)
EXPECTED_BLOCKS = EXPECTED_FAMILIES * len(PROMPT_STYLES) * len(TASKS)
EXPECTED_PATCH_RECORDS = EXPECTED_BLOCKS * RECORDS_PER_BLOCK
assert RECORDS_PER_BLOCK == 51
assert EXPECTED_BLOCKS == 120 and EXPECTED_PATCH_RECORDS == 6120
manifest_sha256 = sha256_file(MANIFEST_PATH)
protocol_sha256 = sha256_file(PROTOCOL_LOCK_PATH)

capture_identity = {
    'schema_version': 'route-relocation-capture-1.0',
    'model_id': MODEL_ID, 'resolved_revision': MODEL_REVISION,
    'dtype': MODEL_DTYPE,
    'dataset_fingerprint': example_config.dataset_fingerprint(),
    'manifest_sha256': manifest_sha256,
    'claim_registry_sha256': protocol_sha256,
    'prompt_styles': list(PROMPT_STYLES),
    'tasks': list(TASKS), 'roles': list(SCENE_ROLES),
    'interventions': list(INTERVENTIONS),
    'scan_sites': [list(site) for site in SCAN_SITES],
    'records_per_block': RECORDS_PER_BLOCK,
    'expected_blocks': EXPECTED_BLOCKS,
    'expected_patch_records': EXPECTED_PATCH_RECORDS,
    'single_example_interventions': True,
    'within_prompt_style_patching_only': True,
    'layer_path': LAYER_PATH, 'seed': SEED,
}
capture_config_path = OUTPUT_ROOT / 'capture_config.json'
if capture_config_path.is_file():
    previous = json.loads(capture_config_path.read_text(encoding='utf-8'))
    assert previous.get('identity') == capture_identity, (
        'Existing capture has a different identity. Use a new version.'
    )
else:
    write_json_atomic(capture_config_path, {'identity': capture_identity})

def block_path(family_index, prompt_style, task):
    return SHARD_DIR / (
        f'block_{family_index:04d}_{prompt_style}_{task}.json'
    )

completed = 0
outer = tqdm(
    list(enumerate(relocation_manifest)),
    desc='Qwen-27B route relocation', unit='family',
)
for family_index, row in outer:
    for prompt_style in PROMPT_STYLES:
        for task in TASKS:
            path = block_path(family_index, prompt_style, task)
            if path.is_file():
                payload = json.loads(path.read_text(encoding='utf-8'))
                if (
                    payload.get('family_id') == row['family_id']
                    and payload.get('prompt_style') == prompt_style
                    and payload.get('task') == task
                    and payload.get('manifest_sha256') == manifest_sha256
                    and payload.get('protocol_sha256') == protocol_sha256
                    and len(payload.get('patch_records', [])) == RECORDS_PER_BLOCK
                ):
                    completed += 1
                    continue

            bundle = make_task_bundle(row, task, prompt_style)
            prepared_by_role = {
                role: prepare_examples([example])
                for role, example in bundle['examples'].items()
            }
            masks = assert_aligned(prepared_by_role)
            scores = {}
            hidden = {}
            for role in SCENE_ROLES:
                scores[role], hidden[role] = forward_capture(
                    prepared_by_role[role]
                )
            conditions = []
            for layer_index, scope, site_label in SCAN_SITES:
                states = {
                    'self_base': hidden['base'][layer_index],
                    'mirror_source': hidden['mirror_source'][layer_index],
                    'cue_source': hidden['cue_source'][layer_index],
                }
                for intervention, intervention_hidden in states.items():
                    conditions.append({
                        'family_id': row['family_id'],
                        'prompt_style': prompt_style, 'task': task,
                        'condition': f'{prompt_style}_{intervention}_{site_label}',
                        'intervention': intervention,
                        'layer_index': layer_index, 'scope': scope,
                        'site_label': site_label,
                        'intervention_hidden': intervention_hidden,
                        'patch_mask': masks[scope],
                    })
            patch_records = []
            inner = tqdm(
                conditions,
                desc=f'{family_index:02d} {prompt_style} {task}',
                unit='site', leave=False,
            )
            for condition in inner:
                prepared_base = prepare_examples([bundle['examples']['base']])
                patch_records.extend(
                    forward_intervention_chunk(prepared_base, [condition])
                )
            assert len(patch_records) == RECORDS_PER_BLOCK
            payload = {
                'schema_version': 'route-relocation-block-1.0',
                'family_id': row['family_id'],
                'prompt_style': prompt_style, 'task': task,
                'manifest_sha256': manifest_sha256,
                'protocol_sha256': protocol_sha256,
                'family': row,
                'baseline': {
                    'semantics': list(bundle['semantics']),
                    'values': bundle['values'], 'scores': scores,
                },
                'patch_records': patch_records,
            }
            temporary = path.with_suffix('.json.tmp')
            temporary.write_text(
                json.dumps(payload, sort_keys=True) + '\n', encoding='utf-8'
            )
            temporary.replace(path)
            completed += 1

assert completed == EXPECTED_BLOCKS
valid_blocks = 0
actual_records = 0
for path in SHARD_DIR.glob('block_*.json'):
    payload = json.loads(path.read_text(encoding='utf-8'))
    if (
        payload.get('manifest_sha256') == manifest_sha256
        and payload.get('protocol_sha256') == protocol_sha256
    ):
        valid_blocks += 1
        actual_records += len(payload.get('patch_records', []))
assert valid_blocks == EXPECTED_BLOCKS
assert actual_records == EXPECTED_PATCH_RECORDS
write_json_atomic(OUTPUT_ROOT / 'capture_complete.json', {
    'blocks': valid_blocks, 'patch_records': actual_records,
    'families': EXPECTED_FAMILIES,
    'manifest_sha256': manifest_sha256,
    'protocol_sha256': protocol_sha256,
    'peak_gpu_gb': round(torch.cuda.max_memory_allocated() / 2**30, 3),
})
print('Capture complete:', valid_blocks, 'blocks /', actual_records, 'records')


## 4. Relocation tests, robustness, and decision


In [ ]:
%%cell_progress 12/17 Release GPU and assemble route-relocation estimands
cell_stage('Releasing model memory and assembling block shards')
import gc
del adapter
gc.collect()
torch.cuda.empty_cache()

def semantic_score(record, semantic):
    semantics = list(record['candidate_semantics'])
    scores = np.asarray(record['candidate_logprobs'], dtype=float)
    return float(scores[semantics.index(semantic)])

def semantic_contrast(record, goal, reference):
    return semantic_score(record, goal) - semantic_score(record, reference)

def gold_margin(record, semantic):
    semantics = list(record['candidate_semantics'])
    scores = np.asarray(record['candidate_logprobs'], dtype=float)
    index = semantics.index(semantic)
    return float(scores[index] - np.max(np.delete(scores, index)))

def centered_error(left_record, right_record):
    left = np.asarray(left_record['candidate_logprobs'], dtype=float)
    right = np.asarray(right_record['candidate_logprobs'], dtype=float)
    left -= left.mean()
    right -= right.mean()
    return float(np.max(np.abs(left - right)))

baseline_rows = []
raw_rows = []
for path in tqdm(
    sorted(SHARD_DIR.glob('block_*.json')),
    desc='Assembling route-relocation metrics', unit='block',
):
    payload = json.loads(path.read_text(encoding='utf-8'))
    row = payload['family']
    task = payload['task']
    prompt_style = payload['prompt_style']
    baseline = payload['baseline']
    values = baseline['values']
    scores = baseline['scores']
    base_semantic = values['base']
    mirror_semantic = values['mirror_source']
    natural_mirror_shift = (
        semantic_contrast(scores['mirror_source'], mirror_semantic, base_semantic)
        - semantic_contrast(scores['base'], mirror_semantic, base_semantic)
    )
    parity_errors = []
    if prompt_style == 'phase_original':
        prior = phase_baselines[row['family_id']][task]
        assert list(prior['semantics']) == list(baseline['semantics'])
        for role in SCENE_ROLES:
            assert (
                prior['scores'][role]['candidate_semantics']
                == scores[role]['candidate_semantics']
            )
            parity_errors.append(centered_error(scores[role], prior['scores'][role]))
    baseline_rows.append({
        'family_id': row['family_id'], 'task': task,
        'prompt_style': prompt_style,
        'cue_type': row['cue_type'],
        'base_outcome': row['base_outcome'],
        'cue_source_margin_gap': float(row['cue_source_margin_gap']),
        'base_correct': bool(scores['base']['correct']),
        'mirror_correct': bool(scores['mirror_source']['correct']),
        'cue_correct': bool(scores['cue_source']['correct']),
        'base_prediction': scores['base']['predicted_semantic'],
        'cue_prediction': scores['cue_source']['predicted_semantic'],
        'clean_cue_invariant': bool(
            scores['base']['predicted_semantic']
            == scores['cue_source']['predicted_semantic']
        ),
        'natural_mirror_shift': float(natural_mirror_shift),
        'phase_parity_max_error': (
            float(max(parity_errors)) if parity_errors else np.nan
        ),
    })

    patches = pd.DataFrame(payload['patch_records'])
    assert len(patches) == RECORDS_PER_BLOCK
    for site_label, site_group in patches.groupby('site_label'):
        records = {
            str(record['intervention']): record.to_dict()
            for _, record in site_group.iterrows()
        }
        assert set(records) == set(INTERVENTIONS)
        self_record = records['self_base']
        mirror_record = records['mirror_source']
        cue_record = records['cue_source']
        mirror_target_shift = (
            semantic_contrast(mirror_record, mirror_semantic, base_semantic)
            - semantic_contrast(self_record, mirror_semantic, base_semantic)
        )
        cue_base_margin_change = (
            gold_margin(cue_record, base_semantic)
            - gold_margin(self_record, base_semantic)
        )
        raw_rows.append({
            'family_id': row['family_id'], 'task': task,
            'prompt_style': prompt_style,
            'cue_type': row['cue_type'],
            'base_outcome': row['base_outcome'],
            'cue_source_margin_gap': float(row['cue_source_margin_gap']),
            'site_label': site_label,
            'layer_index': int(site_group.iloc[0]['layer_index']),
            'scope': str(site_group.iloc[0]['scope']),
            'mirror_target_shift': float(mirror_target_shift),
            'cue_base_margin_change': float(cue_base_margin_change),
            'self_replay_base_change': float(
                gold_margin(self_record, base_semantic)
                - gold_margin(scores['base'], base_semantic)
            ),
        })

baseline_frame = pd.DataFrame(baseline_rows)
raw_frame = pd.DataFrame(raw_rows)
assert len(baseline_frame) == EXPECTED_BLOCKS
assert len(raw_frame) == EXPECTED_BLOCKS * len(SCAN_SITES)

scale_rows = []
anchor = baseline_frame[baseline_frame['prompt_style'] == 'phase_original']
for task, subset in anchor.groupby('task'):
    values = np.abs(subset['natural_mirror_shift'].to_numpy(float))
    nonzero = values[values > 1e-6]
    scale = float(np.median(nonzero)) if len(nonzero) else 1.0
    scale_rows.append({
        'task': task, 'phase_task_natural_scale': max(scale, 1e-3),
        'nonzero_natural_mirror_fraction': float((values > 1e-6).mean()),
    })
scale_frame = pd.DataFrame(scale_rows)
paired_frame = raw_frame.merge(scale_frame, on='task', validate='many_to_one')
for column in (
    'mirror_target_shift', 'cue_base_margin_change', 'self_replay_base_change',
):
    paired_frame[f'std_{column}'] = (
        paired_frame[column] / paired_frame['phase_task_natural_scale']
    )
paired_frame['abs_std_cue_leakage'] = (
    paired_frame['std_cue_base_margin_change'].abs()
)
paired_frame['route_index'] = (
    paired_frame['std_mirror_target_shift']
    - paired_frame['abs_std_cue_leakage']
)
paired_frame['cue_margin_matched'] = (
    paired_frame['cue_source_margin_gap'] <= MAX_CUE_DONOR_MARGIN_GAP
)

baseline_frame.to_csv(ANALYSIS_DIR / 'clean_relocation_baselines.csv', index=False)
raw_frame.to_csv(ANALYSIS_DIR / 'raw_relocation_metrics.csv', index=False)
paired_frame.to_csv(
    ANALYSIS_DIR / 'standardized_relocation_metrics.csv', index=False
)
scale_frame.to_csv(ANALYSIS_DIR / 'phase_anchor_scales.csv', index=False)
print('Baseline / route rows:', len(baseline_frame), len(paired_frame))
display(scale_frame)
display(baseline_frame.groupby(['task', 'prompt_style'])[[
    'base_correct', 'mirror_correct', 'cue_correct', 'clean_cue_invariant'
]].mean())
display(baseline_frame[
    baseline_frame['prompt_style'] == 'phase_original'
].groupby('task')['phase_parity_max_error'].agg(['count', 'mean', 'max']))


In [ ]:
%%cell_progress 13/17 Run globally corrected route-relocation tests
cell_stage('Running paired bootstrap and shared-family max-statistic correction')

def seed_for(*values):
    digest = hashlib.sha256(
        (str(SEED) + '|' + '|'.join(map(str, values))).encode('utf-8')
    ).hexdigest()
    return int(digest[:8], 16)

def clean_values(values):
    return pd.to_numeric(
        pd.Series(values), errors='coerce'
    ).dropna().to_numpy(float)

def summarize(values, label):
    values = clean_values(values)
    assert len(values) > 0, label
    rng = np.random.default_rng(seed_for(label))
    indices = rng.integers(
        0, len(values), size=(BOOTSTRAP_REPLICATES, len(values))
    )
    boot = values[indices].mean(axis=1)
    return {
        'n': int(len(values)), 'mean': float(values.mean()),
        'ci_low': float(np.quantile(boot, ALPHA / 2)),
        'ci_high': float(np.quantile(boot, 1 - ALPHA / 2)),
    }

def paired_delta(frame, metric, left_style, right_style):
    pivot = frame.pivot(
        index='family_id', columns='prompt_style', values=metric
    )
    delta = (pivot[left_style] - pivot[right_style]).dropna()
    assert len(delta) == EXPECTED_FAMILIES
    return delta.sort_index()

def paired_contrast(frame, metric, left_style, right_style, label):
    delta = paired_delta(frame, metric, left_style, right_style)
    values = delta.to_numpy(float)
    result = summarize(values, label)
    observed = float(values.mean())
    rng = np.random.default_rng(seed_for(label, 'sign_flip'))
    signs = rng.choice(
        (-1.0, 1.0), size=(PERMUTATION_REPLICATES, len(values))
    )
    null = (signs @ values) / len(values)
    p_value = float(
        (1 + np.sum(np.abs(null) >= abs(observed)))
        / (PERMUTATION_REPLICATES + 1)
    )
    return {
        'left_style': left_style, 'right_style': right_style,
        'estimate': observed, 'permutation_p': p_value, **result,
    }

cell_rows = []
for (task, site_label, prompt_style), subset in paired_frame.groupby([
    'task', 'site_label', 'prompt_style'
]):
    for metric in (
        'route_index', 'std_mirror_target_shift', 'abs_std_cue_leakage',
    ):
        cell_rows.append({
            'task': task, 'site_label': site_label,
            'prompt_style': prompt_style, 'metric': metric,
            **summarize(
                subset[metric],
                f'cell:{task}:{site_label}:{prompt_style}:{metric}',
            ),
        })
cell_tests = pd.DataFrame(cell_rows)

contrast_rows = []
primary_deltas = []
primary_keys = []
for task in TASKS:
    for site_label in PRIMARY_SITES:
        subset = paired_frame[
            (paired_frame['task'] == task)
            & (paired_frame['site_label'] == site_label)
        ]
        for left_style, right_style in (
            ('structured_physics', 'phase_original'),
            ('generic_deliberation', 'phase_original'),
            ('structured_physics', 'generic_deliberation'),
        ):
            row = {
                'task': task, 'site_label': site_label, 'metric': 'route_index',
                **paired_contrast(
                    subset, 'route_index', left_style, right_style,
                    f'route:{task}:{site_label}:{left_style}:{right_style}',
                ),
            }
            contrast_rows.append(row)
            if (
                left_style == 'structured_physics'
                and right_style == 'generic_deliberation'
            ):
                primary_deltas.append(paired_delta(
                    subset, 'route_index', left_style, right_style
                ))
                primary_keys.append((task, site_label))
contrast_tests = pd.DataFrame(contrast_rows)

family_order = list(primary_deltas[0].index)
assert all(list(delta.index) == family_order for delta in primary_deltas)
delta_matrix = np.column_stack([
    delta.to_numpy(float) for delta in primary_deltas
])
observed = delta_matrix.mean(axis=0)
rng = np.random.default_rng(seed_for('global_maxT'))
signs = rng.choice(
    (-1.0, 1.0),
    size=(PERMUTATION_REPLICATES, EXPECTED_FAMILIES),
)
null_means = signs @ delta_matrix / EXPECTED_FAMILIES
max_null = np.max(null_means, axis=1)
maxT_p = np.asarray([
    (1 + np.sum(max_null >= value)) / (PERMUTATION_REPLICATES + 1)
    for value in observed
], dtype=float)
contrast_tests['maxT_p'] = np.nan
for (task, site_label), adjusted in zip(primary_keys, maxT_p):
    mask = (
        (contrast_tests['task'] == task)
        & (contrast_tests['site_label'] == site_label)
        & (contrast_tests['left_style'] == 'structured_physics')
        & (contrast_tests['right_style'] == 'generic_deliberation')
    )
    assert int(mask.sum()) == 1
    contrast_tests.loc[mask, 'maxT_p'] = float(adjusted)

accuracy_rows = []
for task in TASKS:
    subset = baseline_frame[
        baseline_frame['task'] == task
    ][['family_id', 'prompt_style', 'base_correct']].copy()
    subset['accuracy'] = subset['base_correct'].astype(float)
    for left_style, right_style in (
        ('structured_physics', 'phase_original'),
        ('generic_deliberation', 'phase_original'),
        ('structured_physics', 'generic_deliberation'),
    ):
        accuracy_rows.append({
            'task': task,
            **paired_contrast(
                subset, 'accuracy', left_style, right_style,
                f'accuracy:{task}:{left_style}:{right_style}',
            ),
        })
accuracy_tests = pd.DataFrame(accuracy_rows)

cell_tests.to_csv(ANALYSIS_DIR / 'relocation_route_cells.csv', index=False)
contrast_tests.to_csv(ANALYSIS_DIR / 'relocation_route_contrasts.csv', index=False)
accuracy_tests.to_csv(ANALYSIS_DIR / 'relocation_accuracy_contrasts.csv', index=False)
display(contrast_tests[
    (contrast_tests['left_style'] == 'structured_physics')
    & (contrast_tests['right_style'] == 'generic_deliberation')
].sort_values(['maxT_p', 'task', 'site_label']).head(20))
display(accuracy_tests)


In [ ]:
%%cell_progress 14/17 Audit parity, LOO, shortcut type, orientation, and anchors
cell_stage('Applying frozen robustness checks to all 34 primary tests')

def delta_for_site(task, site_label, left_style, right_style):
    subset = paired_frame[
        (paired_frame['task'] == task)
        & (paired_frame['site_label'] == site_label)
    ]
    return paired_delta(subset, 'route_index', left_style, right_style)

family_meta = manifest_frame.set_index('family_id')
robustness_rows = []
for task in TASKS:
    for site_label in PRIMARY_SITES:
        vs_original = delta_for_site(
            task, site_label, 'structured_physics', 'phase_original'
        )
        vs_generic = delta_for_site(
            task, site_label, 'structured_physics', 'generic_deliberation'
        )
        loo_original = [
            float(vs_original.drop(index=family_id).mean())
            for family_id in vs_original.index
        ]
        loo_generic = [
            float(vs_generic.drop(index=family_id).mean())
            for family_id in vs_generic.index
        ]
        conflict_original = {}
        conflict_generic = {}
        orientation_original = {}
        orientation_generic = {}
        for cue_type in ('weight_conflict', 'distance_conflict'):
            ids = family_meta[family_meta['cue_type'] == cue_type].index
            conflict_original[cue_type] = float(
                vs_original.reindex(ids).dropna().mean()
            )
            conflict_generic[cue_type] = float(
                vs_generic.reindex(ids).dropna().mean()
            )
        for outcome in ('left', 'right'):
            ids = family_meta[family_meta['base_outcome'] == outcome].index
            orientation_original[outcome] = float(
                vs_original.reindex(ids).dropna().mean()
            )
            orientation_generic[outcome] = float(
                vs_generic.reindex(ids).dropna().mean()
            )
        site_row = paired_frame[
            (paired_frame['task'] == task)
            & (paired_frame['site_label'] == site_label)
        ].iloc[0]
        robustness_rows.append({
            'task': task, 'site_label': site_label,
            'layer_index': int(site_row['layer_index']),
            'scope': str(site_row['scope']),
            'is_prior_frozen_site': site_label in PRIOR_FROZEN_SITES,
            'loo_min_vs_original': float(min(loo_original)),
            'loo_min_vs_generic': float(min(loo_generic)),
            'both_conflict_types_positive_vs_original': bool(
                all(value > 0 for value in conflict_original.values())
            ),
            'both_conflict_types_positive_vs_generic': bool(
                all(value > 0 for value in conflict_generic.values())
            ),
            'both_orientations_positive_vs_original': bool(
                all(value > 0 for value in orientation_original.values())
            ),
            'both_orientations_positive_vs_generic': bool(
                all(value > 0 for value in orientation_generic.values())
            ),
            'weight_gain_vs_original': conflict_original['weight_conflict'],
            'distance_gain_vs_original': conflict_original['distance_conflict'],
            'weight_gain_vs_generic': conflict_generic['weight_conflict'],
            'distance_gain_vs_generic': conflict_generic['distance_conflict'],
        })
robustness_frame = pd.DataFrame(robustness_rows)

parity_frame = baseline_frame[
    baseline_frame['prompt_style'] == 'phase_original'
][[
    'family_id', 'task', 'phase_parity_max_error'
]].copy()
parity_gate = bool(
    parity_frame['phase_parity_max_error'].max() <= PARITY_TOLERANCE
)
self_replay_audit = paired_frame.groupby([
    'task', 'site_label', 'prompt_style'
])['std_self_replay_base_change'].agg(['count', 'mean', 'max', 'min']).reset_index()
assert np.isfinite(self_replay_audit[['mean', 'max', 'min']].to_numpy(float)).all()

robustness_frame.to_csv(ANALYSIS_DIR / 'relocation_robustness.csv', index=False)
parity_frame.to_csv(ANALYSIS_DIR / 'phase_original_parity.csv', index=False)
self_replay_audit.to_csv(ANALYSIS_DIR / 'self_replay_audit.csv', index=False)
print('Phase-original parity gate:', parity_gate)
display(parity_frame.groupby('task')['phase_parity_max_error'].agg([
    'count', 'mean', 'max'
]))
display(robustness_frame.sort_values([
    'loo_min_vs_generic', 'loo_min_vs_original'
], ascending=False).head(20))


In [ ]:
%%cell_progress 15/17 Save the paper-facing route-relocation figure
cell_stage('Rendering behavior, instruction scan, option scan, and anchor')
import matplotlib.pyplot as plt

style_order = list(PROMPT_STYLES)
style_labels = {
    'phase_original': 'Phase original',
    'generic_deliberation': 'Generic deliberate',
    'structured_physics': 'Structured physics',
}
colors = {
    'phase_original': '#6B7280',
    'generic_deliberation': '#D97706',
    'structured_physics': '#2563EB',
}
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

accuracy = baseline_frame.groupby([
    'task', 'prompt_style'
])['base_correct'].mean().unstack()
x = np.arange(len(TASKS))
width = 0.24
for index, style in enumerate(style_order):
    axes[0, 0].bar(
        x + (index - 1) * width,
        [accuracy.loc[task, style] for task in TASKS],
        width, label=style_labels[style], color=colors[style],
    )
axes[0, 0].set_xticks(x, TASKS)
axes[0, 0].set_ylim(0, 1)
axes[0, 0].axhline(1/3, color='black', linestyle='--', linewidth=.8)
axes[0, 0].set_ylabel('Clean base accuracy')
axes[0, 0].set_title('A. Exact-anchor behavior')
axes[0, 0].legend(frameon=False)

primary = contrast_tests[
    (contrast_tests['left_style'] == 'structured_physics')
    & (contrast_tests['right_style'] == 'generic_deliberation')
].copy()
for axis, scope, title in (
    (axes[0, 1], 'answer_instruction_tokens', 'B. Answer-instruction relocation'),
    (axes[1, 0], 'options_tokens', 'C. Options-token relocation'),
):
    matrix = []
    p_matrix = []
    for task in TASKS:
        values = []
        p_values = []
        for layer in SCAN_LAYERS:
            site = f'l{layer}_' + ('instruction' if scope == 'answer_instruction_tokens' else 'options')
            row = primary[
                (primary['task'] == task)
                & (primary['site_label'] == site)
            ].iloc[0]
            values.append(float(row['estimate']))
            p_values.append(float(row['maxT_p']))
        matrix.append(values)
        p_matrix.append(p_values)
    matrix = np.asarray(matrix, dtype=float)
    limit = max(.10, float(np.max(np.abs(matrix))))
    image = axis.imshow(
        matrix, aspect='auto', cmap='coolwarm', vmin=-limit, vmax=limit
    )
    axis.set_xticks(np.arange(len(SCAN_LAYERS)), SCAN_LAYERS)
    axis.set_yticks(np.arange(len(TASKS)), TASKS)
    axis.set_xlabel('Decoder layer')
    axis.set_title(title)
    for row_index in range(len(TASKS)):
        for column_index in range(len(SCAN_LAYERS)):
            if p_matrix[row_index][column_index] < ALPHA:
                axis.text(column_index, row_index, '*', ha='center', va='center')
    fig.colorbar(image, ax=axis, fraction=.046, pad=.04)

answer = primary[primary['site_label'] == 'l63_answer_anchor'].set_index('task')
values = [float(answer.loc[task, 'estimate']) for task in TASKS]
errors = np.asarray([
    [
        float(answer.loc[task, 'estimate'] - answer.loc[task, 'ci_low'])
        for task in TASKS
    ],
    [
        float(answer.loc[task, 'ci_high'] - answer.loc[task, 'estimate'])
        for task in TASKS
    ],
])
axes[1, 1].bar(np.arange(len(TASKS)), values, color=['#2563EB', '#F97316'])
axes[1, 1].errorbar(
    np.arange(len(TASKS)), values, yerr=errors,
    fmt='none', color='black', capsize=4,
)
axes[1, 1].axhline(0, color='black', linewidth=1)
axes[1, 1].set_xticks(np.arange(len(TASKS)), TASKS)
axes[1, 1].set_ylabel('Structured − generic route gain')
axes[1, 1].set_title(
    'D. Final-answer anchor\n'
    f'phase parity max={parity_frame.phase_parity_max_error.max():.2g}'
)

fig.suptitle(
    'Qwen3.5-27B: globally corrected prompt-conditioned route relocation',
    fontsize=15,
)
fig.tight_layout()
figure_path = FIGURE_DIR / 'qwen27b_route_relocation_audit.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', figure_path)


In [ ]:
%%cell_progress 16/17 Apply the frozen relocation decision tree
cell_stage('Combining global correction, specificity, parity, and robustness')

structured_accuracy = baseline_frame[
    baseline_frame['prompt_style'] == 'structured_physics'
].groupby('task')['base_correct'].mean()
competence_gate = bool(all(
    float(structured_accuracy[task]) >= MIN_STRUCTURED_ACCURACY
    for task in TASKS
))

primary = contrast_tests[
    (contrast_tests['left_style'] == 'structured_physics')
    & (contrast_tests['right_style'] == 'generic_deliberation')
].set_index(['task', 'site_label'])
vs_original = contrast_tests[
    (contrast_tests['left_style'] == 'structured_physics')
    & (contrast_tests['right_style'] == 'phase_original')
].set_index(['task', 'site_label'])
robust = robustness_frame.set_index(['task', 'site_label'])

passing_rows = []
for key, row in primary.iterrows():
    original_row = vs_original.loc[key]
    audit = robust.loc[key]
    passes = bool(
        row['estimate'] > 0
        and row['ci_low'] > 0
        and row['maxT_p'] < ALPHA
        and original_row['estimate'] > 0
        and original_row['ci_low'] > 0
        and audit['loo_min_vs_original'] > 0
        and audit['loo_min_vs_generic'] > 0
        and audit['both_conflict_types_positive_vs_original']
        and audit['both_conflict_types_positive_vs_generic']
        and audit['both_orientations_positive_vs_original']
        and audit['both_orientations_positive_vs_generic']
    )
    if passes:
        passing_rows.append({
            'task': key[0], 'site_label': key[1],
            'layer_index': int(audit['layer_index']),
            'scope': str(audit['scope']),
            'is_prior_frozen_site': bool(audit['is_prior_frozen_site']),
            'structured_minus_generic': float(row['estimate']),
            'structured_minus_generic_ci_low': float(row['ci_low']),
            'maxT_p': float(row['maxT_p']),
            'structured_minus_original': float(original_row['estimate']),
            'structured_minus_original_ci_low': float(original_row['ci_low']),
        })

new_site_rows = [
    row for row in passing_rows if not row['is_prior_frozen_site']
]
anchor_rows = [
    row for row in passing_rows if row['is_prior_frozen_site']
]
structured_vs_original_accuracy = accuracy_tests[
    (accuracy_tests['left_style'] == 'structured_physics')
    & (accuracy_tests['right_style'] == 'phase_original')
]
behavior_gain = bool((
    (structured_vs_original_accuracy['estimate'] > 0)
    & (structured_vs_original_accuracy['ci_low'] > 0)
    & (structured_vs_original_accuracy['permutation_p'] < ALPHA)
).any())

if not parity_gate:
    decision = 'phase_original_parity_failed'
elif competence_gate and new_site_rows:
    decision = 'physics_specific_route_relocation_supported'
elif competence_gate and anchor_rows:
    decision = 'physics_specific_route_enhancement_at_prior_anchor'
elif behavior_gain:
    decision = 'behavior_shift_without_detected_route_relocation'
else:
    decision = 'no_physics_specific_route_relocation_detected'

decision_payload = {
    'schema_version': 'qwen27b-route-relocation-decision-1.0',
    'decision': decision,
    'model_id': MODEL_ID, 'model_revision': MODEL_REVISION,
    'families': EXPECTED_FAMILIES,
    'sites': len(SCAN_SITES),
    'primary_tests': len(SCAN_SITES) * len(TASKS),
    'manifest_sha256': sha256_file(MANIFEST_PATH),
    'claim_registry_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'parity_gate': parity_gate,
    'phase_parity_max_error': float(
        parity_frame['phase_parity_max_error'].max()
    ),
    'competence_gate': competence_gate,
    'behavior_gain': behavior_gain,
    'passing_rows': passing_rows,
    'new_site_rows': new_site_rows,
    'anchor_rows': anchor_rows,
    'structured_accuracy': {
        task: float(structured_accuracy[task]) for task in TASKS
    },
    'best_descriptive_primary_rows': (
        contrast_tests[
            (contrast_tests['left_style'] == 'structured_physics')
            & (contrast_tests['right_style'] == 'generic_deliberation')
        ].sort_values('maxT_p').head(10).to_dict(orient='records')
    ),
    'limitations': [
        'The depth grid is coarse and tests single-layer residual sufficiency.',
        'A null does not exclude a computation distributed across several layers or components.',
        'The audit reuses the frozen phase-map support rather than an independent scene holdout.',
        'Prompt styles are compared within style; activations are not patched across unequal-length prompts.',
        'The experiment does not reveal literal chain of thought or a unique circuit.',
    ],
}
DECISION_PATH = ANALYSIS_DIR / 'route_relocation_decision.json'
write_json_atomic(DECISION_PATH, decision_payload)
print(json.dumps(decision_payload, indent=2))


In [ ]:
%%cell_progress 17/17 Export the relocation claim card and readable report
cell_stage('Writing the decision-aware route-relocation report')

interpretations = {
    'phase_original_parity_failed': (
        'The exact phase anchor did not reproduce the saved forward pass, so '
        'the relocation analysis cannot yet be connected to the phase-map result.'
    ),
    'physics_specific_route_relocation_supported': (
        'The structured prompt produces a globally corrected, physics-specific '
        'route gain at a new frozen layer/token site. This supports relocation '
        'rather than simple repair of the original route.'
    ),
    'physics_specific_route_enhancement_at_prior_anchor': (
        'The structured prompt strengthens a prior frozen route site under the '
        'exact phase anchor, but no new site satisfies the relocation gates.'
    ),
    'behavior_shift_without_detected_route_relocation': (
        'The prompt changes clean behavior, but no frozen layer/token site '
        'shows globally corrected physics-specific route relocation.'
    ),
    'no_physics_specific_route_relocation_detected': (
        'Neither behavior nor the globally corrected depth/token scan supports '
        'physics-specific route relocation on the frozen support.'
    ),
}

report_lines = [
    '# Qwen-27B prompt-conditioned route relocation audit', '',
    f"- Decision: `{decision}`",
    f'- Frozen families: {EXPECTED_FAMILIES}',
    f'- Scan sites: {len(SCAN_SITES)}',
    f'- Globally corrected primary tests: {len(SCAN_SITES) * len(TASKS)}',
    f'- Phase-original parity gate: {parity_gate}',
    f'- Structured competence gate: {competence_gate}',
    f'- Passing sites: {len(passing_rows)}', '',
    '## Interpretation', '', interpretations[decision], '',
    '## Evidence boundary', '',
    'A passing site must beat both the exact phase-original prompt and the '
    'length-matched generic control, survive one-sided max-statistic correction '
    'across every task/site, and pass LOO, shortcut-type, orientation, parity, '
    'and competence gates. Descriptive peaks are not treated as discoveries.', '',
    '## Recommended next step', '',
]
if new_site_rows:
    report_lines.extend([
        'Freeze the passing site and replicate the exact three-prompt contrast '
        'on Gemma-4-12B, followed by Qwen3.5-9B. Do not rescan model-specific '
        'layers without a normalized-depth preregistration.', '',
    ])
elif anchor_rows:
    report_lines.extend([
        'Treat the result as route enhancement rather than relocation and '
        'replicate the anchor effect on an independent model family.', '',
    ])
else:
    report_lines.extend([
        'Stop searching additional single residual sites. Retain the '
        'behavior–mechanism dissociation and, if one further test is needed, '
        'use a prespecified multi-layer/component intervention or explicit '
        'intermediate-output experiment.', '',
    ])

REPORT_PATH = ANALYSIS_DIR / 'route_relocation_report.md'
REPORT_PATH.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')
write_json_atomic(ANALYSIS_DIR / 'paper_claim_card.json', {
    'decision': decision,
    'interpretation': interpretations[decision],
    'passing_rows': passing_rows,
    'parity_gate': parity_gate,
    'nonclaim': (
        'The audit does not identify a unique circuit, literal reasoning trace, '
        'or rule out computation distributed across multiple layers.'
    ),
})
print(REPORT_PATH.read_text(encoding='utf-8'))
print('All outputs:', ANALYSIS_DIR)
